In [ ]:
#%pip install python-dotenv
#%pip install openai
#%pip install scikit-learn

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached openai-2.48.0-py3-none-any.whl.metadata (36 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.16.0-cp314-cp314-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.69.1-py3-none-any.whl.metadata (57 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py

In [1]:
import os
 
from dotenv import load_dotenv
from openai import OpenAI
 
load_dotenv()
 
api_key = os.getenv("DEEPSEEK_API_KEY")
 
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)
 
print(api_key is not None)

True


In [2]:
import sys
import os
from dotenv import load_dotenv
from importlib.metadata import version

load_dotenv()

print(f"Python: {sys.version}")

try:
    import sklearn
    print(f"scikit-learn: {sklearn.__version__} ✓")
except ImportError:
    print("scikit-learn no instalado — ejecuta: pip install scikit-learn")

try:
    import openai
    print(f"openai: {version('openai')} ✓")
except ImportError:
    print("openai no instalado — ejecuta: pip install openai")

api_key = os.getenv("DEEPSEEK_API_KEY")

if api_key:
    print(f"DEEPSEEK_API_KEY: configurada ✓ (termina en ...{api_key[-4:]})")
else:
    print("DEEPSEEK_API_KEY: NO configurada ⚠ — revisa el archivo .env")

Python: 3.14.6 (main, Jul 18 2026, 17:08:46) [MSC v.1944 64 bit (AMD64)]
scikit-learn: 1.9.0 ✓
openai: 2.48.0 ✓
DEEPSEEK_API_KEY: configurada ✓ (termina en ...8626)


In [3]:
from collections import Counter

MENSAJES = [

    # --- horario (10 mensajes) ---
    ("¿A qué hora abren los sábados?", "horario"),
    ("¿Cuál es el horario de atención al cliente?", "horario"),
    ("¿Hasta qué hora atienden los domingos?", "horario"),
    ("¿Trabajan el 20 de julio?", "horario"),
    ("¿Las oficinas abren el lunes festivo?", "horario"),
    ("Necesito saber cuándo abren el banco", "horario"),
    ("¿Hay atención los días festivos?", "horario"),
    ("¿A qué hora cierran la sucursal de El Poblado?", "horario"),
    ("¿Tienen atención los fines de semana?", "horario"),
    ("¿Qué días están disponibles los asesores?", "horario"),

    # --- queja (10 mensajes) ---
    ("Me debitaron dos veces el mismo pago", "queja"),
    ("No reconozco un cobro en mi extracto", "queja"),
    ("Llevo 3 días sin poder usar mi tarjeta débito", "queja"),
    ("El cajero se tragó mi tarjeta y no me la devolvió", "queja"),
    ("Me bloquearon la cuenta sin avisarme", "queja"),
    ("Hice una transferencia y el dinero no llegó", "queja"),
    ("Me cobraron una comisión que no autoricé", "queja"),
    ("Llevo una hora en la línea y nadie me atiende", "queja"),
    ("Mi solicitud de crédito lleva 15 días sin respuesta", "queja"),
    ("El app del banco no me deja ingresar desde ayer", "queja"),

    # --- producto (10 mensajes) ---
    ("¿Cuál es la tasa de interés del crédito de consumo?", "producto"),
    ("¿Qué documentos necesito para abrir una cuenta de ahorros?", "producto"),
    ("¿Cuánto es el cupo mínimo de la tarjeta de crédito?", "producto"),
    ("¿Tienen CDTs a 90 días?", "producto"),
    ("¿Qué beneficios tiene la tarjeta Visa Platinum?", "producto"),
    ("¿Puedo pedir un préstamo si soy independiente?", "producto"),
    ("¿Cuánto tiempo tardan en desembolsar un crédito?", "producto"),
    ("¿Qué seguro incluye la cuenta corriente?", "producto"),
    ("¿Tienen cuenta de nómina para empresas pequeñas?", "producto"),
    ("¿Cuál es el monto máximo del giro internacional?", "producto"),

    # --- saludo (10 mensajes) ---
    ("Hola buenos días, necesito ayuda", "saludo"),
    ("Buenas tardes, ¿me pueden atender?", "saludo"),
    ("Hola, ¿están disponibles?", "saludo"),
    ("Buen día", "saludo"),
    ("Buenos días, tengo una pregunta", "saludo"),
    ("Hola, cómo están", "saludo"),
    ("Buenas noches, requiero asistencia", "saludo"),
    ("Hey, ¿alguien me puede ayudar?", "saludo"),
    ("Hola, estoy aquí", "saludo"),
    ("Bienvenidos, soy cliente nuevo", "saludo"),
]

print(f"Total de mensajes: {len(MENSAJES)}")

conteo = Counter(etiqueta for _, etiqueta in MENSAJES)

for categoria, cantidad in sorted(conteo.items()):
    print(f"{categoria}: {cantidad}")

Total de mensajes: 40
horario: 10
producto: 10
queja: 10
saludo: 10


In [4]:
# Celda 3 — Clasificador simbólico

def clasificar_simbolico(mensaje: str) -> str:
    """
    Clasifica la intención del mensaje usando reglas IF-THEN.
    No usa ningún dato de entrenamiento.
    """

    texto = mensaje.lower()

    # ── REGLAS PARA 'horario' ────────────────────────────────
    palabras_horario = [
        "hora", "horario", "abren", "cierran", "atienden", "atención",
        "disponibles", "festivo", "sábado", "domingo", "lunes", "días"
    ]
    if any(p in texto for p in palabras_horario):
        return "horario"

    # ── REGLAS PARA 'queja' ──────────────────────────────────
    palabras_queja = [
        "no reconozco", "cobro", "debit", "bloquearon", "bloqueada",
        "no llegó", "no me deja", "se tragó", "sin respuesta",
        "no pude", "no puedo", "error", "falla", "problema",
        "dos veces", "no funciona", "días sin", "sin avisarme",
        "transferencia", "comisión", "queja", "reclamo"
    ]
    if any(p in texto for p in palabras_queja):
        return "queja"

    # ── REGLAS PARA 'producto' ───────────────────────────────
    palabras_producto = [
        "tasa", "interés", "crédito", "cuenta", "tarjeta", "cdt",
        "préstamo", "documentos", "cupo", "beneficios", "seguro",
        "nómina", "giro", "desembolsar", "monto", "visa", "platinum",
        "independiente", "ahorros", "corriente"
    ]
    if any(p in texto for p in palabras_producto):
        return "producto"

    # ── REGLAS PARA 'saludo' ─────────────────────────────────
    palabras_saludo = [
        "hola", "buenos", "buenas", "buen día", "hey", "bienvenidos"
    ]
    if any(p in texto for p in palabras_saludo):
        return "saludo"

    return "desconocido"

In [5]:
# Celda 4 — Evaluación simbólico

import time
from collections import Counter

def evaluar_clasificador(fn_clasificar, mensajes, nombre):
    """Evalúa cualquier función clasificadora y devuelve métricas."""
    correctos = 0
    errores = []
    tiempos = []

    for texto, esperado in mensajes:
        t0 = time.perf_counter()
        pred = fn_clasificar(texto)
        t1 = time.perf_counter()

        tiempos.append(t1 - t0)
        if pred == esperado:
            correctos += 1
        else:
            errores.append((texto, esperado, pred))

    precision = correctos / len(mensajes)
    latencia_ms = sum(tiempos) / len(tiempos) * 1000

    print(f"\n{'-' * 55}")
    print(f"  {nombre}")
    print(f"{'-' * 55}")
    print(f"  Precisión:   {correctos}/{len(mensajes)} = {precision*100:.1f}%")
    print(f"  Latencia:    {latencia_ms:.3f} ms (promedio por mensaje)")
    print(f"\n  Errores ({len(errores)}):")
    for texto, esperado, pred in errores:
        print(f"    X [{esperado}]->[{pred}] | '{texto[:50]}'")

    return {
        "precision": precision,
        "latencia_ms": latencia_ms,
        "errores": errores
    }


resultados_simbolico = evaluar_clasificador(
    clasificar_simbolico,
    MENSAJES,
    "SIMBÓLICO — Reglas IF-THEN"
)


-------------------------------------------------------
  SIMBÓLICO — Reglas IF-THEN
-------------------------------------------------------
  Precisión:   32/40 = 80.0%
  Latencia:    0.003 ms (promedio por mensaje)

  Errores (8):
    X [horario]->[desconocido] | '¿Trabajan el 20 de julio?'
    X [queja]->[horario] | 'Llevo 3 días sin poder usar mi tarjeta débito'
    X [queja]->[horario] | 'Llevo una hora en la línea y nadie me atiende'
    X [queja]->[horario] | 'Mi solicitud de crédito lleva 15 días sin respuest'
    X [producto]->[horario] | '¿Tienen CDTs a 90 días?'
    X [saludo]->[horario] | 'Hola buenos días, necesito ayuda'
    X [saludo]->[horario] | 'Hola, ¿están disponibles?'
    X [saludo]->[horario] | 'Buenos días, tengo una pregunta'


## ¿Cuantos errores cometio el sistema simbolico? ------- 8 Errores.
## ¿Que tiene en comun los mensajes de fallo? ------- En su mayoria eran mensajes sobre el horario.
## ¿Cuantas reglas adicionales necesitas para cumplir esos casos? ------- Solo se deberia agregar 1 regla adicional para que todas tengan una clasificacion.

In [6]:
# Celda 5 — Preparación del dataset para sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Separar textos y etiquetas
textos = [m[0] for m in MENSAJES]
etiquetas = [m[1] for m in MENSAJES]

# División 75% entrenamiento / 25% prueba — estratificada (mismas proporciones por clase)
X_train, X_test, y_train, y_test = train_test_split(
    textos, etiquetas,
    test_size=0.25,
    random_state=42,
    stratify=etiquetas
)

print(f"Entrenamiento: {len(X_train)} mensajes")
print(f"Prueba:        {len(X_test)} mensajes")

from collections import Counter
print(f"Clases en prueba: {Counter(y_test)}")

Entrenamiento: 30 mensajes
Prueba:        10 mensajes
Clases en prueba: Counter({'horario': 3, 'queja': 3, 'producto': 2, 'saludo': 2})


In [7]:
# Celda 6 — Entrenamiento

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # uni-gramas y bi-gramas
    min_df=1,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=1.0
)

clf.fit(X_train_vec, y_train)

print("Modelo entrenado ✓")
print(f"Vocabulario: {len(vectorizer.vocabulary_)} términos")

Modelo entrenado ✓
Vocabulario: 274 términos


In [8]:
# Celda 7 — Evaluación estadístico

def clasificar_estadistico(mensaje: str) -> str:
    vec = vectorizer.transform([mensaje])
    return clf.predict(vec)[0]

resultados_estadistico = evaluar_clasificador(
    clasificar_estadistico,
    MENSAJES,
    "ESTADÍSTICO — TF-IDF + Logistic Regression"
)

# Reporte detallado por clase
y_pred_test = clf.predict(X_test_vec)

print(f"\n{'-' * 55}")
print("  Reporte en conjunto de prueba (10 mensajes):")
print(f"{'-' * 55}")

print(classification_report(y_test, y_pred_test))


-------------------------------------------------------
  ESTADÍSTICO — TF-IDF + Logistic Regression
-------------------------------------------------------
  Precisión:   39/40 = 97.5%
  Latencia:    0.295 ms (promedio por mensaje)

  Errores (1):
    X [saludo]->[producto] | 'Bienvenidos, soy cliente nuevo'

-------------------------------------------------------
  Reporte en conjunto de prueba (10 mensajes):
-------------------------------------------------------
              precision    recall  f1-score   support

     horario       1.00      1.00      1.00         3
    producto       0.67      1.00      0.80         2
       queja       1.00      1.00      1.00         3
      saludo       1.00      0.50      0.67         2

    accuracy                           0.90        10
   macro avg       0.92      0.88      0.87        10
weighted avg       0.93      0.90      0.89        10



## ¿El modelo estadístico cometió más o menos errores que el simbólico? ------- Este comete menos errores comparado al simbolico.
## ¿Los errores son los mismos o diferentes? ------- Son parecidos, ya que evidencia como un mensaje se le dio una calsificacion erronea.
## El modelo se entrenó con 30 ejemplos (7-8 por clase). ¿Es suficiente? ¿Qué pasaría con 300? ------- Este llegaria a ser mejor gracia al entrenamiento adicional.

In [9]:
# Celda 8 — Clasificador generativo con DeepSeek V4-Flash

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError("No se encontró DEEPSEEK_API_KEY en el archivo .env")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

CATEGORIAS_VALIDAS = {
    "horario",
    "queja",
    "producto",
    "saludo"
}


def clasificar_generativo(mensaje: str) -> str:
    """
    Clasifica la intención del mensaje utilizando DeepSeek V4-Flash.
    Devuelve exactamente una de las categorías válidas.
    """

    respuesta = client.chat.completions.create(
        model="deepseek-v4-flash",
        temperature=0,
        max_tokens=20,

        # Desactivar el modo thinking.
        # Para clasificación no necesitamos razonamiento,
        # solo la categoría final.
        extra_body={
            "thinking": {
                "type": "disabled"
            }
        },

        messages=[
            {
                "role": "system",
                "content": (
                    "Eres el clasificador de intenciones del chatbot de un "
                    "banco colombiano.\n\n"

                    "Clasifica el siguiente mensaje en UNA de estas categorías:\n"
                    "- horario\n"
                    "- queja\n"
                    "- producto\n"
                    "- saludo\n\n"

                    "Responde SOLO con la palabra de la categoría, "
                    "sin explicación."
                )
            },
            {
                "role": "user",
                "content": mensaje
            }
        ]
    )

    resultado = respuesta.choices[0].message.content

    # Comprobar si la respuesta está vacía
    if resultado is None:
        return "desconocido"

    resultado = resultado.strip().lower()

    # Normalizar la respuesta
    for categoria in CATEGORIAS_VALIDAS:
        if categoria in resultado:
            return categoria

    return "desconocido"

In [10]:
# Celda 9 — Evaluación generativo (puede tardar ~1-2 minutos)

print("Llamando a la API de DeepSeek para cada mensaje...")
print("(esto tarda ~1-2 minutos – 40 llamadas secuenciales)\n")

resultados_generativo = evaluar_clasificador(
    clasificar_generativo,
    MENSAJES,
    "GENERATIVO — DeepSeek"
)

Llamando a la API de DeepSeek para cada mensaje...
(esto tarda ~1-2 minutos – 40 llamadas secuenciales)


-------------------------------------------------------
  GENERATIVO — DeepSeek
-------------------------------------------------------
  Precisión:   40/40 = 100.0%
  Latencia:    1046.412 ms (promedio por mensaje)

  Errores (0):


In [16]:
# Celda 10 — Tabla comparativa

# Costo estimado por 1,000 mensajes (precios aproximados julio 2026)
# Claude Haiku ~ $0.25 por millón de tokens de entrada, ~$1.25 por millón de salida
# Estimando ~50 tokens entrada + 3 tokens salida por mensaje:
# (50 * 0.25/1_000_000 + 3 * 1.25/1_000_000) * 1000 = $0.016 por 1,000 mensajes
COSTO_POR_1K_HAIKU = 0.016  # USD estimado

print("=" * 60)
print(" TABLA COMPARATIVA — Tres Paradigmas de IA ")
print("=" * 60)
print(f"{'Métrica':<28} {'Simbólico':>10} {'Estadístico':>12} {'Generativo':>11}")
print("-" * 60)

# Precisión
p_sim = resultados_simbolico["precision"] * 100
p_est = resultados_estadistico["precision"] * 100
p_gen = resultados_generativo["precision"] * 100
print(f"{'Precisión (%)':<28} {p_sim:>9.1f}% {p_est:>11.1f}% {p_gen:>10.1f}%")

# Latencia
l_sim = resultados_simbolico["latencia_ms"]
l_est = resultados_estadistico["latencia_ms"]
l_gen = resultados_generativo["latencia_ms"]
print(f"{'Latencia promedio (ms)':<28} {l_sim:>9.3f} {l_est:>11.3f} {l_gen:>10.1f}")

# Datos de entrenamiento
print(f"{'Datos requeridos':<28} {'0':>10} {'30 msgs':>12} {'0':>11}")

# Costo por 1,000 mensajes
print(f"{'Costo / 1,000 msgs (USD)':<28} {'$0.00':>10} {'$0.00':>12} {f'~${0.02}':>11}")

# Explicabilidad
print(f"{'Explicabilidad':<28} {'Total':>10} {'Parcial':>12} {'Baja':>11}")

# Requiere reentrenamiento
print(f"{'Reentrenar al cambiar dominio':<28} {'No':>10} {'Sí':>12} {'No':>11}")

print("=" * 60)

# Errores cualitativos

print("\n Mensajes que fallaron en cada enfoque:")
print("=" * 60)

for nombre, res in [
    ("Simbólico", resultados_simbolico),
    ("Estadístico", resultados_estadistico),
    ("Generativo", resultados_generativo)
]:
    errores = res["errores"]

    if errores:
        print(f"\n {nombre} ({len(errores)} errores):")
        for texto, esperado, pred in errores:
            print(f"   [{esperado}]→[{pred}]: '{texto[:45]}'")
    else:
        print(f"\n {nombre}: 0 errores ✓")


 TABLA COMPARATIVA — Tres Paradigmas de IA 
Métrica                       Simbólico  Estadístico  Generativo
------------------------------------------------------------
Precisión (%)                     80.0%        97.5%        0.0%
Latencia promedio (ms)           0.003       0.464     1089.8
Datos requeridos                      0      30 msgs           0
Costo / 1,000 msgs (USD)          $0.00        $0.00      ~$0.02
Explicabilidad                    Total      Parcial        Baja
Reentrenar al cambiar dominio         No           Sí          No

 Mensajes que fallaron en cada enfoque:

 Simbólico (8 errores):
   [horario]→[desconocido]: '¿Trabajan el 20 de julio?'
   [queja]→[horario]: 'Llevo 3 días sin poder usar mi tarjeta débito'
   [queja]→[horario]: 'Llevo una hora en la línea y nadie me atiende'
   [queja]→[horario]: 'Mi solicitud de crédito lleva 15 días sin res'
   [producto]→[horario]: '¿Tienen CDTs a 90 días?'
   [saludo]→[horario]: 'Hola buenos días, necesito ayuda'
 

In [17]:
# BONO — Clasificador híbrido: simbólico primero, LLM como respaldo

def clasificar_hibrido(mensaje: str, umbral_confianza: float = 0.8) -> tuple[str, str]:
    """
    Estrategia híbrida: usa el modelo estadístico si la confianza es alta,
    de lo contrario delega al LLM.
    Retorna (etiqueta, fuente) donde fuente es 'estadistico' o 'llm'.
    """

    # Obtener probabilidades del modelo estadístico
    vec = vectorizer.transform([mensaje])
    proba = clf.predict_proba(vec)[0]
    confianza_max = proba.max()
    pred_estadistico = clf.classes_[proba.argmax()]

    if confianza_max >= umbral_confianza:
        return pred_estadistico, "estadistico"
    else:
        # Confianza baja → delegar al LLM
        pred_llm = clasificar_generativo(mensaje)
        return pred_llm, "llm"


# Prueba con los mensajes más ambiguos

mensajes_ambiguos = [
    "Mi aplicación no me dejó ingresar ayer",    # ¿queja o producto?
    "Buenas tardes, tengo un problema",          # ¿saludo o queja?
    "Quiero información sobre préstamos",        # ¿producto o saludo?
    "¡Trabajan mañana que es festivo?",          # ¿horario claramente?
    "¡Llevo esperando demasiado tiempo!",        # ¿queja sin palabras clave típicas?
]

print("Clasificador Híbrido — Mensajes Ambiguos:")
print("-" * 60)

for mensaje in mensajes_ambiguos:
    etiqueta, fuente = clasificar_hibrido(mensaje)
    print(f"[{fuente:12}] {etiqueta:10} | '{mensaje}'")

Clasificador Híbrido — Mensajes Ambiguos:
------------------------------------------------------------
[llm         ] desconocido | 'Mi aplicación no me dejó ingresar ayer'
[llm         ] desconocido | 'Buenas tardes, tengo un problema'
[llm         ] desconocido | 'Quiero información sobre préstamos'
[llm         ] desconocido | '¡Trabajan mañana que es festivo?'
[llm         ] desconocido | '¡Llevo esperando demasiado tiempo!'


## Pruebas

In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError("No se encontró DEEPSEEK_API_KEY en el archivo .env")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1"
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    temperature=0.7,
    max_tokens=20,
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Explícame los números primos."}
    ]
)

print(response.choices[0].message.content)

In [20]:
from pprint import pprint

pprint(response.model_dump())

{'choices': [{'finish_reason': 'length',
              'index': 0,
              'logprobs': None,
              'message': {'annotations': None,
                          'audio': None,
                          'content': '',
                          'function_call': None,
                          'reasoning_content': '¡Claro! Los números primos son '
                                               'un concepto fundamental en '
                                               'matemáticas. Voy a explicarlo',
                          'refusal': None,
                          'role': 'assistant',
                          'tool_calls': None}}],
 'created': 1784910671,
 'id': 'caaf9f1d-9752-41ff-9cd8-a5a08397852e',
 'model': 'deepseek-v4-flash',
 'moderation': None,
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
 'usage': {'completion_tokens': 20,
           'completion_tokens_details': {'accepted_prediction_to

Norma 1 (L1) es la resta del valor absoluto para diferencial el valor real de lo esperado, ya que sin este el valor tiend a 0
Elevacion cuadrado (L2)
MSE vs MAE Valores atipicos
Muchos outlayes = cuadrado
No outlayes = Norma 1